# 📖 Lab 6: High-Demand Events (Deep Dive)

**Non-functional requirement:** *The system should handle 10 million users trying to book one event.*

When Taylor Swift tickets go on sale, the seat map goes stale in seconds. Users click seats that are already taken, submit payment for gone tickets, and get frustrated. This deep dive explores two approaches:

## 🏗️ Architecture — Before (Labs 1-4)

```
┌────────┐       ┌─────────────┐       ┌────────────────┐       ┌──────────────┐
│ Client │──────>│ API Gateway │──────>│ Event Service   │──────>│  PostgreSQL  │
│  😡    │       └─────────────┘       ├────────────────┤       └──────────────┘
│ stale  │                             │Booking Service  │──────>┌──────────────┐
│ seat   │                             │  reserve/confirm│       │    Redis     │
│ map!   │                             └───────┬────────┘       │  Ticket Lock │
└────────┘                                     v                └──────────────┘
                                            Stripe
           10M users all see the seat map at once → chaos
```

## 🏗️ Architecture — After (SSE + Virtual Queue)

```
┌────────┐       ┌──────────────────┐       ┌────────────────┐       ┌──────────┐
│        │──────>│ Virtual Queue    │──────> │                │       │          │
│        │       │ (Redis ZSET)     │ admit  │                │       │          │
│ Client │       │                  │ + JWT  │ Booking Service│──────>│PostgreSQL│
│        │<──SSE─│ position updates │        │                │       │          │
│        │       └──────────────────┘        │                │──────>│          │
│        │                                   └───────┬────────┘       └──────────┘
│        │<──SSE── seat map updates                  │               ┌──────────┐
│        │         (Redis Pub/Sub)                   v               │  Redis   │
└────────┘                                        Stripe             │Lock+Queue│
                                                                     └──────────┘
           Users wait in line → admitted in batches → fair experience
```

## The Two Approaches

| # | Approach | Level | Key Idea |
|---|----------|-------|----------|
| 1 | **SSE real-time seat updates** | 🟡 Good (Senior) | Push seat changes to all viewers instantly |
| 2 | **Virtual waiting queue** | 🟢 Great (Staff) | Gate access to the booking page — solve the business problem, not just the technical one |

## Learning Objectives

- Implement Server-Sent Events (SSE) for real-time seat status broadcasting
- Understand Redis Pub/Sub as the messaging backbone
- Build a virtual waiting queue backed by Redis sorted sets
- See how the queue protects both the system AND the user experience
- Understand the senior vs staff engineering mindset difference

## 🛠️ Setup

```bash
cd system-designs/ticketmaster
docker-compose up -d
```

Select the **"Ticketmaster (Python)"** kernel.

In [ ]:
import redis
import psycopg2
import psycopg2.extras
import threading
import time
import json
import queue

DB_CONFIG = {
    "host": "localhost",
    "port": 5433,
    "user": "demo",
    "password": "demo",
    "database": "ticketmaster",
}

def get_connection():
    return psycopg2.connect(**DB_CONFIG)

redis_client = redis.Redis(host="localhost", port=6380, decode_responses=True)
redis_client.flushdb()

conn = get_connection()
cur = conn.cursor()
cur.execute("SELECT COUNT(*) FROM tickets WHERE event_id = 1 AND status = 'available'")
print(f"✅ PostgreSQL: {cur.fetchone()[0]} available tickets for Event 1")
cur.close()
conn.close()
print(f"✅ Redis: {'connected' if redis_client.ping() else 'FAILED'}")

## 🟡 Good Approach: Real-Time Seat Updates with SSE + Redis Pub/Sub

**Problem:** The seat map goes stale. Users see "available" seats that are already taken.

**Solution:** Use **Server-Sent Events (SSE)** to push seat status changes to all connected clients in real-time. When someone books or reserves a ticket, the Booking Service publishes the change via **Redis Pub/Sub**, and every client viewing that event's seat map gets the update instantly.

```
Booking Service ──publish──> Redis Pub/Sub (channel: event:1:seats)
                                  │
                         ┌────────┼────────┐
                         v        v        v
                      Client A  Client B  Client C
                      (SSE)     (SSE)     (SSE)
                      seat map updates instantly
```

### Why SSE over WebSocket?

We only need **server → client** communication (push updates). SSE is simpler — single HTTP connection, auto-reconnect, no protocol upgrade. WebSocket would work too, but it's overkill when there's no client → server data flow on this channel.

Let's simulate this. Since we're in a notebook (not a real HTTP server), we'll use threads and queues to simulate the SSE pattern.

In [ ]:
# ── Redis Pub/Sub: the messaging backbone ──

CHANNEL = "event:1:seats"

def publish_seat_change(ticket_id: int, new_status: str, section: str, row: str, seat: int):
    """
    Called by the Booking Service after a ticket is reserved/booked.
    Publishes the change to Redis Pub/Sub so all SSE clients get it.
    """
    message = json.dumps({
        "ticketId": ticket_id,
        "status": new_status,
        "section": section,
        "row": row,
        "seat": seat,
        "timestamp": time.time(),
    })
    redis_client.publish(CHANNEL, message)


def simulate_sse_client(client_name: str, event_log: list, stop_event: threading.Event):
    """
    Simulates a client connected via SSE.
    In production, this would be an HTTP endpoint streaming `text/event-stream`.
    Here, we subscribe to Redis Pub/Sub and collect messages.
    """
    # Each subscriber needs its own Redis connection
    sub_client = redis.Redis(host="localhost", port=6380, decode_responses=True)
    pubsub = sub_client.pubsub()
    pubsub.subscribe(CHANNEL)

    for message in pubsub.listen():
        if stop_event.is_set():
            break
        if message["type"] == "message":
            data = json.loads(message["data"])
            event_log.append({
                "client": client_name,
                "received": data,
                "latency_ms": round((time.time() - data["timestamp"]) * 1000, 1),
            })

    pubsub.unsubscribe()
    pubsub.close()
    sub_client.close()


# Start 3 "SSE clients" listening for seat changes
stop = threading.Event()
client_logs = {name: [] for name in ["Alice", "Bob", "Carol"]}
client_threads = []

for name in client_logs:
    t = threading.Thread(target=simulate_sse_client, args=(name, client_logs[name], stop), daemon=True)
    t.start()
    client_threads.append(t)

time.sleep(0.5)  # Let subscribers connect

# Simulate 3 bookings happening
print("🎫 Simulating real-time seat changes...\n")

bookings = [
    (101, "reserved", "FLOOR", "A", 5),
    (102, "booked",   "FLOOR", "A", 6),
    (103, "reserved", "LOWER", "B", 10),
]

for ticket_id, status, section, row, seat in bookings:
    publish_seat_change(ticket_id, status, section, row, seat)
    print(f"  📢 Published: Ticket {ticket_id} → {status} ({section}-{row}-Seat {seat})")
    time.sleep(0.2)

time.sleep(0.5)  # Let messages propagate
stop.set()
time.sleep(0.3)

# Show what each client received
print(f"\n{'='*60}")
print("📥 Messages received by each SSE client:")
print(f"{'='*60}")

for name, logs in client_logs.items():
    print(f"\n  {name} received {len(logs)} updates:")
    for log in logs:
        d = log["received"]
        print(f"    Ticket {d['ticketId']}: {d['status']} ({d['section']}-{d['row']}-Seat {d['seat']}) — latency: {log['latency_ms']}ms")

print(f"\n💡 All 3 clients saw all 3 seat changes in real-time.")
print(f"   In production, this would be SSE events pushed over HTTP.")
print(f"   The client JS would update the seat map SVG instantly.")

### 🤔 Why SSE Alone Isn't Enough for Taylor Swift

Real-time updates solve the **stale seat map** problem. But for 10M users competing for 20,000 seats:

```
10,000,000 users watching    →    Every seat booked in ~2 seconds
20,000 seats                 →    Seat map flashes from full to empty instantly
                             →    Users see a blur of disappearing seats
                             →    Clicking anything results in "already taken"
                             →    😡 Terrible experience, even with real-time updates
```

The real insight: **real-time updates are a technical solution to a technical problem.** But the actual problem is a **business problem**: too many users competing for too few seats creates a bad experience regardless of how fast you update the UI.

A staff engineer thinks: *"What if users don't ALL see the seat map at the same time?"*

## 🟢 Great Approach: Virtual Waiting Queue

Instead of letting 10 million users slam the booking page simultaneously, we **gate access** with a virtual queue. Users wait in line and get admitted in controlled batches.

```
10M users arrive
      │
      v
┌─────────────────────────────┐
│     Virtual Queue           │
│   (Redis Sorted Set)        │
│                             │
│  #1 Alice    ──────────┐    │
│  #2 Bob      ──────────┤    │  Admit batch of N users
│  #3 Carol    ──────────┤    │  when capacity available
│  #4 Dave     waiting   │    │
│  #5 Eve      waiting   │    │
│  ...                   │    │
│  #10M        waiting   │    │
└─────────────────────────────┘
                         │
                         v
              ┌────────────────────┐
              │   Booking Page     │
              │   (admitted only)  │
              │                    │
              │   Seat map +       │
              │   real-time SSE    │
              │   updates          │
              └────────────────────┘
```

### How it works

1. User requests the booking page → added to a Redis sorted set (score = timestamp)
2. SSE connection established → pushes position updates ("You are #4,521...")
3. Queue service periodically admits N users from the front
4. Admitted users get added to `admitted:{eventId}` set in Redis (with TTL)
5. Booking Service checks this set → only admitted users can reserve tickets
6. If admitted user doesn't book in time, their admission expires, and the next user in queue gets in

In [ ]:
# ── Virtual Queue Service ──

QUEUE_KEY = "queue:event:1"
ADMITTED_KEY = "admitted:event:1"
ADMITTED_TTL = 300  # 5 minutes to complete booking once admitted
BATCH_SIZE = 5      # Admit 5 users at a time (small for demo; 100-500 in production)


def join_queue(user_id: str, event_id: int = 1) -> dict:
    """
    User requests to view the booking page → placed in the queue.
    Returns their position. In production, an SSE connection is established.
    """
    queue_key = f"queue:event:{event_id}"
    timestamp = time.time()

    # ZADD NX: only add if not already in queue (prevents re-joining)
    added = redis_client.zadd(queue_key, {user_id: timestamp}, nx=True)

    # Get position (0-indexed rank)
    position = redis_client.zrank(queue_key, user_id)
    total = redis_client.zcard(queue_key)

    return {
        "userId": user_id,
        "position": position + 1,
        "totalInQueue": total,
        "status": "queued" if added else "already_in_queue",
    }


def get_queue_position(user_id: str, event_id: int = 1) -> dict:
    """Check current position in queue (called periodically via SSE)."""
    queue_key = f"queue:event:{event_id}"
    position = redis_client.zrank(queue_key, user_id)

    if position is None:
        # Maybe already admitted?
        if redis_client.sismember(f"admitted:event:{event_id}", user_id):
            return {"userId": user_id, "status": "admitted", "message": "🎉 You're in! Go to the booking page."}
        return {"userId": user_id, "status": "not_found"}

    return {
        "userId": user_id,
        "position": position + 1,
        "totalInQueue": redis_client.zcard(queue_key),
        "status": "waiting",
    }


def admit_next_batch(event_id: int = 1) -> list[str]:
    """
    Dequeue the next BATCH_SIZE users and mark them as admitted.
    Called periodically or when capacity becomes available.
    """
    queue_key = f"queue:event:{event_id}"
    admitted_key = f"admitted:event:{event_id}"

    # Get the first BATCH_SIZE users (lowest scores = earliest arrivals)
    users = redis_client.zrange(queue_key, 0, BATCH_SIZE - 1)

    if not users:
        return []

    # Move them from queue to admitted set
    pipe = redis_client.pipeline()
    for user_id in users:
        pipe.zrem(queue_key, user_id)
        pipe.sadd(admitted_key, user_id)
    pipe.execute()

    return users


def is_admitted(user_id: str, event_id: int = 1) -> bool:
    """Booking Service calls this to verify user has been admitted."""
    return redis_client.sismember(f"admitted:event:{event_id}", user_id)


print("✅ Queue service functions defined.")

## 🧪 Simulating the Queue Experience

Let's simulate 20 users arriving for a hot event. We'll admit them in batches and watch the queue drain.

In [ ]:
# Clean slate
redis_client.delete(QUEUE_KEY, ADMITTED_KEY)

# 20 users arrive within a short window
print("🏃 20 users arriving for The Eras Tour tickets...\n")

for i in range(1, 21):
    result = join_queue(f"user_{i:03d}")
    if i <= 5 or i > 18:
        print(f"  User {i:3d} joined → position #{result['position']}")
    elif i == 6:
        print(f"  ... (users 6-18 joining)")

print(f"\n📊 Queue size: {redis_client.zcard(QUEUE_KEY)} users waiting")
print(f"   Admitted so far: {redis_client.scard(ADMITTED_KEY)}")

# Simulate the queue draining in batches
print(f"\n{'='*60}")
print(f"⏳ Admitting users in batches of {BATCH_SIZE}...")
print(f"{'='*60}")

batch_num = 0
while redis_client.zcard(QUEUE_KEY) > 0:
    batch_num += 1
    admitted = admit_next_batch()
    remaining = redis_client.zcard(QUEUE_KEY)
    total_admitted = redis_client.scard(ADMITTED_KEY)

    print(f"\n  Batch {batch_num}: admitted {admitted}")
    print(f"    Queue remaining: {remaining} | Total admitted: {total_admitted}")

    # Show what user_010 sees (somewhere in the middle)
    status = get_queue_position("user_010")
    if status["status"] == "waiting":
        print(f"    → user_010 sees: 'You are #{status['position']} of {status['totalInQueue']} in line'")
    elif status["status"] == "admitted":
        print(f"    → user_010 sees: '{status['message']}'")

    time.sleep(0.3)  # Simulates batch interval

print(f"\n✅ All users admitted! Queue is empty.")
print(f"   Total admitted: {redis_client.scard(ADMITTED_KEY)}")

## 🔒 Booking Service: Admission Check

The queue is only useful if the Booking Service **enforces** it. Unadmitted users should be rejected from the booking page entirely.

In [ ]:
def reserve_with_queue_check(user_id: str, event_id: int, ticket_id: int) -> dict:
    """
    Enhanced Booking Service: checks queue admission before allowing reservation.
    Only users who have been admitted through the queue can reserve tickets.
    """
    # Step 1: Check if this event requires queue admission
    # (in production, this would be an admin-set flag per event)
    high_demand = True  # Simulating a high-demand event

    if high_demand and not is_admitted(user_id, event_id):
        return {
            "success": False,
            "error": "You haven't been admitted yet. Please wait in the queue.",
        }

    # Step 2: Normal reservation flow (from Lab 4)
    lock_key = f"ticket:{ticket_id}"
    acquired = redis_client.set(lock_key, user_id, nx=True, ex=300)

    if not acquired:
        return {"success": False, "error": f"Ticket {ticket_id} is already reserved"}

    return {
        "success": True,
        "message": f"Ticket {ticket_id} reserved for {user_id}!",
    }


# Test: admitted user vs unadmitted user
print("🔒 Testing admission enforcement:\n")

# user_001 was admitted (from the batch above)
result_admitted = reserve_with_queue_check("user_001", event_id=1, ticket_id=999)
print(f"  user_001 (admitted):    {result_admitted}")

# user_sneaky was never in the queue
result_sneaky = reserve_with_queue_check("user_sneaky", event_id=1, ticket_id=998)
print(f"  user_sneaky (not admitted): {result_sneaky}")

# Cleanup
redis_client.delete("ticket:999")

## 🔐 How Admission Actually Works in Production

You might be thinking: *"Checking a Redis set on every request is fine, but how does the client prove they've been admitted? Can't someone just skip the queue?"*

In production, the admission flow uses a **signed, short-lived token** (JWT or similar) that the client must present on every booking request. Here's the full mechanism:

### The Flow

```
Queue Service                          Client                          Booking Service
────────────                          ──────                          ───────────────
1. User admitted from queue
2. Generate signed admission token:
   JWT {
     userId: "alice",
     eventId: 1,
     exp: NOW + 10 min,              ───token via SSE──>
     iat: NOW                        3. Client stores token
   }
                                     4. Client redirects
                                        to booking page
                                        with token

                                     5. POST /bookings/reserve
                                        Authorization: Bearer <token>
                                                                       6. Verify JWT signature
                                                                       7. Check: exp > NOW?
                                                                       8. Check: userId matches?
                                                                       9. (Optional) Check Redis
                                                                          admitted set as backup
                                                                       10. Allow reservation ✅
```

### Three Layers of Authorization

| Layer | What | Why |
|-------|------|-----|
| **JWT token** (primary) | Signed token with `userId`, `eventId`, `exp` | Client-side proof. Stateless verification — no Redis call needed on every request. Can't be forged without the signing key. |
| **Redis admitted set** (backup) | `admitted:{eventId}` set | Server-side source of truth. Used to revoke admission (e.g., remove user if they misbehave). Also used for counting admitted users. |
| **TTL / expiration** | JWT `exp` claim + Redis key TTL | Automatic cleanup. If user doesn't book within 10 minutes, token expires and admission is revoked. |

### Why JWT and not just a session cookie?

| Session Cookie | JWT |
|---------------|-----|
| Requires server-side session store | Stateless — Booking Service verifies signature without any DB/Redis call |
| Tied to one server (sticky sessions) or needs shared session store | Works across any Booking Service instance (horizontal scaling) |
| Revocation is easy (delete session) | Revocation needs Redis check (or short TTL) |

For this system, JWT is ideal because the Booking Service is horizontally scaled — any instance can verify the token without hitting a shared session store. The short TTL (10 minutes) means revocation is rarely needed.

### What the token contains

```json
{
  "sub": "user_001",          // who
  "eventId": 1,               // which event they're admitted for
  "iat": 1710000000,          // when they were admitted
  "exp": 1710000600,          // when admission expires (+10 min)
  "type": "queue_admission"   // token type (distinguishes from auth tokens)
}
```

Let's implement this.

In [ ]:
import hmac
import hashlib
import base64

# In production, this would be an environment variable / secret manager
SIGNING_SECRET = "super-secret-key-do-not-hardcode"
ADMISSION_TTL = 600  # 10 minutes in production


def create_admission_token(user_id: str, event_id: int) -> str:
    """
    Queue Service creates this when a user is admitted.
    A lightweight signed token (simplified JWT for demo).
    """
    payload = json.dumps({
        "sub": user_id,
        "eventId": event_id,
        "iat": int(time.time()),
        "exp": int(time.time()) + ADMISSION_TTL,
        "type": "queue_admission",
    })
    payload_b64 = base64.urlsafe_b64encode(payload.encode()).decode()

    # Sign the payload so it can't be tampered with
    signature = hmac.new(
        SIGNING_SECRET.encode(), payload_b64.encode(), hashlib.sha256
    ).hexdigest()

    return f"{payload_b64}.{signature}"


def verify_admission_token(token: str, expected_user: str, expected_event: int) -> dict:
    """
    Booking Service calls this on every request.
    Verifies signature, expiration, and that user/event match.
    No Redis or DB call needed — fully stateless.
    """
    try:
        payload_b64, signature = token.rsplit(".", 1)
    except ValueError:
        return {"valid": False, "error": "Malformed token"}

    # Verify signature (prevents forgery)
    expected_sig = hmac.new(
        SIGNING_SECRET.encode(), payload_b64.encode(), hashlib.sha256
    ).hexdigest()

    if not hmac.compare_digest(signature, expected_sig):
        return {"valid": False, "error": "Invalid signature — token was tampered with"}

    # Decode payload
    payload = json.loads(base64.urlsafe_b64decode(payload_b64))

    # Check expiration
    if payload["exp"] < time.time():
        return {"valid": False, "error": f"Token expired {int(time.time() - payload['exp'])}s ago"}

    # Check user and event match
    if payload["sub"] != expected_user:
        return {"valid": False, "error": "Token belongs to a different user"}
    if payload["eventId"] != expected_event:
        return {"valid": False, "error": "Token is for a different event"}

    remaining = int(payload["exp"] - time.time())
    return {"valid": True, "userId": payload["sub"], "eventId": payload["eventId"],
            "remainingSeconds": remaining}


# ── Updated admit_next_batch: now returns tokens ──

def admit_next_batch_with_tokens(event_id: int = 1) -> list[dict]:
    """
    Dequeues users and generates admission tokens for each.
    In production, these tokens are pushed to clients via SSE.
    """
    queue_key = f"queue:event:{event_id}"
    admitted_key = f"admitted:event:{event_id}"

    users = redis_client.zrange(queue_key, 0, BATCH_SIZE - 1)
    if not users:
        return []

    results = []
    pipe = redis_client.pipeline()
    for user_id in users:
        pipe.zrem(queue_key, user_id)
        pipe.sadd(admitted_key, user_id)
        token = create_admission_token(user_id, event_id)
        results.append({"userId": user_id, "token": token})
    pipe.execute()

    return results


# ── Updated reserve: verifies JWT token ──

def reserve_with_token(user_id: str, event_id: int, ticket_id: int, token: str) -> dict:
    """
    Booking Service: verifies the admission token before allowing reservation.
    Stateless verification — no Redis call needed for auth.
    """
    # Step 1: Verify the admission token (stateless — just crypto)
    verification = verify_admission_token(token, user_id, event_id)
    if not verification["valid"]:
        return {"success": False, "error": verification["error"]}

    # Step 2: Normal reservation flow
    lock_key = f"ticket:{ticket_id}"
    acquired = redis_client.set(lock_key, user_id, nx=True, ex=300)

    if not acquired:
        return {"success": False, "error": f"Ticket {ticket_id} is already reserved"}

    return {
        "success": True,
        "message": f"Ticket {ticket_id} reserved for {user_id}!",
        "timeRemaining": f"{verification['remainingSeconds']}s left to complete booking",
    }


print("✅ Token-based admission functions defined.")

### 🧪 Token Demo: Admission → Token → Booking

In [ ]:
# Reset
redis_client.flushdb()

print("🎬 Full Token-Based Admission Flow\n")
print("="*60)

# Step 1: Users join the queue
print("\n📌 Step 1: Users join the queue")
for i in range(1, 11):
    join_queue(f"user_{i:03d}")
print(f"   10 users in queue")

# Step 2: Admit first batch — now with tokens!
print(f"\n📌 Step 2: Admit batch (with tokens)")
batch = admit_next_batch_with_tokens(event_id=1)
for entry in batch:
    token_preview = entry["token"][:40] + "..."
    print(f"   {entry['userId']} → token: {token_preview}")

# Step 3: Let's look at what's inside Alice's token
alice_token = batch[0]["token"]
print(f"\n📌 Step 3: Decoding Alice's token")
payload_b64 = alice_token.rsplit(".", 1)[0]
payload = json.loads(base64.urlsafe_b64decode(payload_b64))
print(f"   {json.dumps(payload, indent=4)}")
print(f"   Expires in: {payload['exp'] - int(time.time())}s")

# Step 4: Alice uses her token to reserve
print(f"\n📌 Step 4: Alice reserves a ticket with her token")
result = reserve_with_token("user_001", event_id=1, ticket_id=42, token=alice_token)
print(f"   → {result}")

# Step 5: Sneaky user tries with no token
print(f"\n📌 Step 5: Sneaky user tries to forge a booking")
result = reserve_with_token("hacker", event_id=1, ticket_id=43, token="fakepayload.fakesignature")
print(f"   → {result}")

# Step 6: Bob (user_002) tries to use Alice's token
print(f"\n📌 Step 6: Bob tries to use Alice's token")
result = reserve_with_token("user_002", event_id=1, ticket_id=43, token=alice_token)
print(f"   → {result}")

# Step 7: Alice tries to book for a different event
print(f"\n📌 Step 7: Alice tries for event 2 (not admitted)")
result = reserve_with_token("user_001", event_id=2, ticket_id=50, token=alice_token)
print(f"   → {result}")

# Cleanup
redis_client.delete("ticket:42")

print(f"\n{'='*60}")
print(f"\n🔐 Security summary:")
print(f"   ✅ Valid token + correct user + correct event → allowed")
print(f"   ❌ No token → rejected")
print(f"   ❌ Forged token → invalid signature")
print(f"   ❌ Someone else's token → user mismatch")
print(f"   ❌ Token for different event → event mismatch")
print(f"   ❌ Expired token → time's up, back to the queue")

In [ ]:
# Reset
redis_client.flushdb()

print("🎬 Full Flow: Queue → Token → Book (complete user journey)\n")
print("="*60)

# Step 1: Users flood in
print("\n📌 Step 1: 15 users arrive for Taylor Swift tickets")
for i in range(1, 16):
    join_queue(f"user_{i:03d}")
print(f"   Queue size: {redis_client.zcard(QUEUE_KEY)}")

# Step 2: Alice checks position
print(f"\n📌 Step 2: user_001 (Alice) checks position via SSE")
pos = get_queue_position("user_001")
print(f"   → 'You are #{pos['position']} of {pos['totalInQueue']} in line'")

# Step 3: Alice tries to book with no token
print(f"\n📌 Step 3: Alice tries to skip the queue (no token)")
result = reserve_with_token("user_001", event_id=1, ticket_id=1, token="no.token")
print(f"   → ❌ {result['error']}")

# Step 4: First batch admitted — tokens delivered via SSE
print(f"\n📌 Step 4: System admits first batch of {BATCH_SIZE} (tokens sent via SSE)")
batch = admit_next_batch_with_tokens()
tokens = {entry["userId"]: entry["token"] for entry in batch}
for entry in batch:
    print(f"   ✅ {entry['userId']} admitted")

# Step 5: Alice books with her token
print(f"\n📌 Step 5: Alice uses her admission token to reserve")
result = reserve_with_token("user_001", event_id=1, ticket_id=1, token=tokens["user_001"])
print(f"   → {result['message']}")
print(f"   ⏰ {result['timeRemaining']}")

# Step 6: Dave is still waiting
print(f"\n📌 Step 6: user_010 (Dave) checks position")
pos = get_queue_position("user_010")
print(f"   → 'You are #{pos['position']} of {pos['totalInQueue']} in line'")

# Step 7: Next batch
print(f"\n📌 Step 7: Next batch admitted")
batch2 = admit_next_batch_with_tokens()
for entry in batch2:
    print(f"   ✅ {entry['userId']} admitted")
pos = get_queue_position("user_010")
if pos["status"] == "waiting":
    print(f"   user_010 still waiting: position #{pos['position']}")
else:
    print(f"   user_010: {pos['message']}")

print(f"\n{'='*60}")
print(f"\n💡 The complete flow:")
print(f"   1. User joins queue → gets position updates via SSE")
print(f"   2. System admits batch → sends signed JWT token via SSE")
print(f"   3. Client stores token, redirects to booking page")
print(f"   4. Every booking request includes token in Authorization header")
print(f"   5. Booking Service verifies token (stateless, no DB call)")
print(f"   6. Token expires after 10 min → user must re-queue")

## 🧹 Cleanup

In [ ]:
redis_client.flushdb()
print("✅ Redis cleaned up.")

## ✅ Summary: Senior vs Staff Thinking

| | 🟡 Good (Senior) | 🟢 Great (Staff) |
|-|-------------------|-------------------|
| **Approach** | SSE real-time seat updates | Virtual waiting queue |
| **Solves** | Stale seat map (technical problem) | Overwhelming UX (business problem) |
| **User sees** | Seats disappearing in real-time | "You are #4,521 in line" with position updates |
| **System load** | 10M simultaneous connections | Controlled batches of N users |
| **Complexity** | Redis Pub/Sub + SSE | Redis sorted set + SSE + admission check |

### The Staff Engineer Insight

> *"The best solution isn't always technically harder. Sometimes it's recognizing that the real problem is a business problem, not a technical one."*

A senior engineer sees "seat map is stale" → pushes real-time updates. Technically correct, but the UX is still terrible with 10M users.

A staff engineer sees "10M users competing for 20K seats is inherently a bad experience" → gates access with a queue. Fewer users see the seat map at any time = each user has a fair, relaxed experience.

### Implementation Details

| Component | Technology | Purpose |
|-----------|-----------|---------|
| Queue | Redis sorted set (`ZADD`, `ZRANGE`, `ZREM`) | FIFO ordering by timestamp |
| Position updates | SSE (Server-Sent Events) | Push position changes to waiting users |
| Admission tracking | Redis set (`admitted:{eventId}`) | Booking Service checks before allowing reservation |
| Seat map updates | Redis Pub/Sub → SSE | Real-time seat changes for admitted users viewing the map |

### Challenges

| Challenge | Mitigation |
|-----------|-----------|
| Long wait times cause frustration | Push real-time position + estimated wait time via SSE |
| Admitted user doesn't book | TTL on admission — expires, next user gets in |
| Queue fairness | Redis sorted set guarantees FIFO ordering by arrival time |
| Bot/scalper abuse | Rate limiting + CAPTCHA at queue entry point |

**Pattern reference:** See `patterns/real-time-updates/` for SSE and WebSocket implementations